TODO: natively vectorize as many functions as possible to avoid iterations and speed up computation.

In [1]:
from typing import Tuple
import re
from enum import Enum
import numpy as np
import pandas as pd
from wordfreq import word_frequency

WORD_LEN = 5
VALID_RE = re.compile(rf'[a-z]{{{WORD_LEN}}}')
RE_VEC = np.vectorize(VALID_RE.fullmatch)
FREQ_VEC = np.vectorize(lambda word: word_frequency(word, 'en', minimum=1e-8), otypes=[float])

class Square(Enum):
    """Define square colors in an enumeration."""
    BLACK = '\U00002B1B'
    YELLOW = '\U0001F7E8'
    GREEN = '\U0001F7E9'

def import_words(fname: str) -> Tuple[np.ndarray, np.ndarray]:
    """Import and validate words from file."""
    words = np.loadtxt(fname, dtype=str)
    assert np.all(RE_VEC(words))
    assert words.dtype == f'<U{WORD_LEN}'
    chars = np.empty((words.shape[0], WORD_LEN), dtype='<U1')
    for idx, word in enumerate(words):
        chars[idx,:] = list(word)
    return words, chars

words, chars = import_words('words.txt')
df = pd.DataFrame(data=chars, index=words)
df

,0,1,2,3,4
aahed,a,a,h,e,d
aalii,a,a,l,i,i
aapas,a,a,p,a,s
aargh,a,a,r,g,h
aarti,a,a,r,t,i
...,...,...,...,...,...
zuzim,z,u,z,i,m
zygal,z,y,g,a,l
zygon,z,y,g,o,n
zymes,z,y,m,e,s


We need ways to operate with the information that Wordle gives back to us.
1. Given a guess and an answer, what do the squares look like?
2. Given a guess and response squares, what candidates match?

The first operation is defined below in `wordle_compare`. The second operations is defined in `is_match`. Both expect numpy arrays of chars.

In [2]:
def wordle_compare(guess: np.ndarray, answer: np.ndarray) -> np.ndarray:
    """Generate wordle response to given 2 words."""
    assert guess.shape == answer.shape
    sz = guess.shape[0]
    squares = np.full((sz,), Square.BLACK)

    """
    Green is the easiest case to handle by position.
    The mask gm is used to remove them in further processing.
    """
    not_green = (guess != answer)
    squares[~not_green] = Square.GREEN

    """
    We need to get the number (n) of times a guess letter in
    a non-green spot occurs in expected. From there, we color
    the first (up to) n occurences of the letter yellow in guess.
    """
    for cand in np.unique(guess[not_green]):
        cand_count = np.count_nonzero(answer[not_green] == cand)
        for i in range(sz):
            if cand_count == 0:
                break
            if squares[i] == Square.GREEN or guess[i] != cand:
                continue
            squares[i] = Square.YELLOW
            cand_count -= 1

    return squares

In [3]:
from collections import Counter

def is_match(guess: np.ndarray, candidate: np.ndarray, squares: np.ndarray):
    """Given a guess and its response, decide it the candidate is a match."""
    assert guess.shape == candidate.shape == squares.shape

    # Validate that all green square positions match.
    gm = (squares == Square.GREEN)
    if np.any(guess[gm] != candidate[gm]):
        return False
    
    """
    1. The yellow masking must not create matches positionally.
    2. Assuming (1) is checked, the yellow guess letters are a
    multi-subset of the non-green candidate letters.
    """
    ym = (squares == Square.YELLOW)
    if np.any(guess[ym] == candidate[ym]):
        return False
    if not Counter(guess[ym]) <= Counter(candidate[~gm]):
        return False
    
    """
    1. The black masking must not create matches positionally.
    2. The black guess letters that do not appear in yellow positions
    must not be contained in the non-green candidate letters.
    """
    bm = (squares == Square.BLACK)
    if np.any(guess[bm] == candidate[bm]):
        return False
    non_yellow_blacks = set(guess[bm]) - set(guess[ym])
    candidate_non_greens = set(candidate[~gm])
    return not non_yellow_blacks.intersection(candidate_non_greens)

Play a game of Worlde below. Record your guesses and responses as you make them and rerun the following cells to refresh the suggestions.

In [4]:
def convert_str_to_squares(abbrev: str) -> np.ndarray:
    """Convenience wrapper to save myself some copy pasting."""
    squares = []
    for letter in abbrev:
        assert letter in ('g', 'y', 'b')
        if letter == 'g':
            squares.append(Square.GREEN)
        elif letter == 'y':
            squares.append(Square.YELLOW)
        else:
            squares.append(Square.BLACK)
    return np.array(squares)

In [5]:
guesses = [
    'salet',
    'penis',
    'would',
    'posse',
]
responses = [
    'ybbyb',
    'gybby',
    'bgbbb',
    'ggggg',
]

Iteratively filter the full list of words based on guesses and their square responses until we arrive at a subset of candidates. We can sort our suggestions to give the most likely candidates by using word frequency.

In [6]:
filtered = df.copy()
filtered['freq'] = FREQ_VEC(filtered.index)
for guess, response in zip(guesses, responses):
    squares = convert_str_to_squares(response)
    gs_np = df.loc[guess].to_numpy()
    mask = [is_match(gs_np, row.drop('freq').to_numpy(), squares) \
            for _, row in filtered.iterrows()]
    filtered = filtered[mask]
    reccs = filtered.sort_values(by='freq', ascending=False)
    print(reccs[:5], '=' * 35, sep='\n')

       0  1  2  3  4      freq
house  h  o  u  s  e  0.000513
needs  n  e  e  d  s  0.000234
issue  i  s  s  u  e  0.000170
weeks  w  e  e  k  s  0.000155
guess  g  u  e  s  s  0.000148
       0  1  2  3  4          freq
purse  p  u  r  s  e  8.910000e-06
prose  p  r  o  s  e  5.250000e-06
posse  p  o  s  s  e  1.510000e-06
poesy  p  o  e  s  y  4.570000e-08
prese  p  r  e  s  e  4.070000e-08
       0  1  2  3  4          freq
posse  p  o  s  s  e  1.510000e-06
poesy  p  o  e  s  y  4.570000e-08
poyse  p  o  y  s  e  1.000000e-08
       0  1  2  3  4      freq
posse  p  o  s  s  e  0.000002


Once you have the solution, visually ascertain that the produced squares match the game and validate our own function.

In [7]:
expected = 'posse'

for guess, response in zip(guesses, responses):
    gs_np = df.loc[guess].to_numpy()
    exp_np = df.loc[expected].to_numpy()
    squares = wordle_compare(gs_np, exp_np)
    print(guess, ''.join(map(lambda sq: sq.value, squares)))
    assert np.all(squares == convert_str_to_squares(response))
    assert is_match(gs_np, exp_np, squares)

salet 🟨⬛⬛🟨⬛
penis 🟩🟨⬛⬛🟨
would ⬛🟩⬛⬛⬛
posse 🟩🟩🟩🟩🟩


Calculate the entropy of each possible guess and square permutation. This will be stored in an array of shape (`NUM_WORDS` , $3^N$) where $N$ = `WORD_LEN`. We will assume a uniform probability distribution over the possible words. The probability $P(s | w)$ of getting square pattern $s$ after guessing word $w \in W$ is simply the proportion of words $c$ in the corpus such that `is_match(w, c, s)`. We define the information gained by this guess (measured in bits) as:

```math
I(w, s) = - \log_2 P(s | w) = - \log_2 \left( \frac{\text{matches}(w, s)}{|W|} \right)
```

We want to choose the word that maximizes the expected information gain. That is, the entropy $H$ given by the following sum over all square patterns:

```math
H(w) = - \sum_s P(s | w) \cdot \log_2 P(s | w)
```